# M4.11 · Zaman serisi — ARMA/ARIMA'dan RNN ve WaveNet'e

Rastgele süreç ve zaman serisi, katı ve geniş anlamda durağanlık, ARMA (otoregresif + hareketli ortalama), fark alma ve ARIMA, mevsimsel SARIMA; Yule–Walker denklemleri, otokorelasyon ve kısmi otokorelasyon ile model belirleme, en küçük kareler AR tahmini. Ardından ders alıştırması 6'nın beş modeli: tek nöronlu doğrusal model, tek rekürent nöron, üç katmanlı RNN, 8 adımı birden tahmin eden dizi→vektör ve her adımda tahmin eden dizi→dizi (nedensel) RNN; naif tahmin ve yinelemeli tahmin karşılaştırmaları. Son olarak nedensel genişletilmiş konvolüsyonlarla WaveNet, ergodiklik ve uyarlamalı (LMS) filtreler.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-11-timeseries.html](https://m-gonul.github.io/pytorch-lab/m4-11-timeseries.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_11_timeseries.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t_diff():\n        ensure(need(ns, \"difference\")([3, 5, 7, 9, 11]) == [2, 2, 2, 2], \"doğrusal trend\")\n        ensure(ns[\"difference\"]([1, 4, 9, 16, 25, 36], 2) == [2, 2, 2, 2], \"ikinci dereceden trend\")\n        close(need(ns, \"undifference\")([3, 5, 7, 9, 11], [2, 2], 1), [13, 15], what=\"ARIMA d = 1\")\n        close(ns[\"undifference\"]([1, 4, 9, 16, 25, 36], [2, 2, 2], 2), [49, 64, 81], what=\"d = 2: kareler sürer\")\n        p = need(ns, \"arma_predict\")([1.0, 2.0, 0.5], [0.5], [0.4])\n        close(p, [0, 0.9, 1.44, 0.25 + 0.4 * (0.5 - 1.44)], what=\"ARMA(1,1) özyinelemesi\")\n    R_.test(\"fark alma ve ARMA\", t_diff)\n\n    def t_yw():\n        r = need(ns, \"ar_acf\")([0.6, 0.3], 5)\n        close(r[:3], [1, 0.6 / 0.7, 0.6 * 0.6 / 0.7 + 0.3], what=\"AR(2) ρ\")\n        close(ns[\"ar_acf\"]([0.8], 4), [1, 0.8, 0.64, 0.512, 0.4096], what=\"AR(1): ρ_k = φ^k\")\n        r3 = ns[\"ar_acf\"]([0.3, -0.2, 0.4], 6)\n        close(r3[1], 0.3 * r3[0] - 0.2 * r3[1] + 0.4 * r3[2], what=\"AR(3) Yule–Walker k = 1\")\n        close(r3[5], 0.3 * r3[4] - 0.2 * r3[3] + 0.4 * r3[2], what=\"AR(3) özyineleme\")\n    R_.test(\"Yule–Walker\", t_yw)\n\n    def t_cnt():\n        P = need(ns, \"rnn_params\")\n        ensure(P([1]) == 3 and P([16, 16, 1]) == 834 and P([16, 16]) + 136 == 952, \"Keras sayımı\")\n        ensure(P([16, 16, 1], double_bias=True) == 834 + 33, \"PyTorch: katman başına +h\")\n        ensure(need(ns, \"receptive_field\")(2, [2 ** i for i in range(10)] * 2) == 2047, \"WaveNet\")\n        rows = [[k, str(d), ns[\"receptive_field\"](k, d)] for k, d in ((2, [1, 1, 1, 1]), (2, [1, 2, 4, 8]), (3, [1, 2, 4, 8]), (2, [2 ** i for i in range(10)]), (2, [2 ** i for i in range(10)] * 3))]\n        R_.table(\"Nedensel konvolüsyon yığınlarının receptive field'ı\", [\"k\", \"genişletmeler\", \"RF\"], [[a, b if len(b) < 30 else \"1…512 ×\" + str(len(eval(b)) // 10), c] for a, b, c in rows],\n                 note=\"Genişletmesiz yığında RF katman sayısıyla doğrusal, ikiye katlanan genişletmelerle üstel büyür: 10 katmanlık 1…512 bloğu 1024 adım görür. WaveNet bu bloğu birkaç kez yığar.\")\n    R_.test(\"parametre ve RF sayımı\", t_cnt)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef _ar(phi, n, seed, sigma=1.0):\n    rng = np.random.default_rng(seed)\n    e = rng.normal(0, sigma, n + 500)\n    y = np.zeros(n + 500)\n    for t in range(len(y)):\n        y[t] = e[t] + sum(p * y[t - 1 - i] for i, p in enumerate(phi) if t - 1 - i >= 0)\n    return y[500:]\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüler\", lambda: ([max_loops(src, 0, func=f) for f in (\"acf\", \"lag_matrix\", \"fit_ar_ls\", \"fit_ar_yw\")], max_loops(src, 1, func=\"pacf\"), max_loops(src, 1, func=\"forecast_ar\")))\n    y = _ar([0.6, 0.3], 40000, 0)\n\n    def t_fit():\n        X, t = need(ns, \"lag_matrix\")(np.arange(6.), 2)\n        ensure(X.tolist() == [[1, 0], [2, 1], [3, 2], [4, 3]] and t.tolist() == [2, 3, 4, 5], \"gecikme matrisi\")\n        z = np.random.default_rng(1).normal(size=50)\n        close(need(ns, \"acf\")(z, 3)[1], np.sum((z[:-1] - z.mean()) * (z[1:] - z.mean())) / np.sum((z - z.mean()) ** 2), what=\"acf tanımı\")\n        r = ns[\"acf\"](y, 3)\n        close(r[1:3], [0.6 / 0.7, 0.6 * 0.6 / 0.7 + 0.3], atol=0.02, what=\"örnek ACF ≈ kuram\")\n        close(need(ns, \"fit_ar_ls\")(y, 2), [0.6, 0.3], atol=0.02, what=\"LS\")\n        close(need(ns, \"fit_ar_yw\")(y, 2), [0.6, 0.3], atol=0.02, what=\"Yule–Walker\")\n        f = need(ns, \"forecast_ar\")(np.array([1.0, 2.0]), np.array([0.6, 0.3]), 3)\n        close(f, [1.5, 0.6 * 1.5 + 0.3 * 2, 0.6 * (0.6 * 1.5 + 0.6) + 0.3 * 1.5], what=\"yinelemeli tahmin\")\n    R_.test(\"AR tahmini\", t_fit)\n\n    def t_id():\n        y1 = _ar([0.7], 20000, 2)\n        y3 = _ar([0.5, -0.3, 0.3], 20000, 3)\n        rows = []\n        for name, s in ((\"AR(1) φ = 0.7\", y1), (\"AR(2) (0.6, 0.3)\", y), (\"AR(3) (0.5, −0.3, 0.3)\", y3)):\n            a = ns[\"acf\"](s, 6)\n            p = need(ns, \"pacf\")(s, 6)\n            rows.append([name, \" \".join(f\"{v:+.2f}\" for v in a[1:]), \" \".join(f\"{v:+.2f}\" for v in p[1:])])\n        p1, p3 = ns[\"pacf\"](y1, 5), ns[\"pacf\"](y3, 5)\n        ensure(abs(p1[1] - 0.7) < 0.03 and np.all(np.abs(p1[2:]) < 0.03), \"AR(1) PACF 1'den sonra kesilir\")\n        ensure(abs(p3[3] - 0.3) < 0.03 and np.all(np.abs(p3[4:]) < 0.03), \"AR(3) PACF 3'ten sonra kesilir\")\n        R_.table(\"Box–Jenkins model belirleme: ACF söner, PACF p'de kesilir\", [\"süreç\", \"ACF (gecikme 1–6)\", \"PACF (gecikme 1–6)\"], rows,\n                 note=\"AR(p) sürecinin ACF'si üstel/sönümlü salınımla söner, PACF'si p'den sonra sıfırdır (gecikme p'deki PACF = φ_p). MA(q) için tersi geçerlidir: ACF q'dan sonra kesilir. Gürültü bandı ≈ ±2/√n.\")\n    R_.test(\"ACF / PACF\", t_id)\n",
 "d": "import time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    S = need(ns, \"generate_time_series\")(5000, 48, 0)\n    Xtr, Xte = S[:4000, :40], S[4000:, :40]\n    y1tr, y1te = S[:4000, 40], S[4000:, 40]\n    y8tr, y8te = S[:4000, 40:, 0], S[4000:, 40:, 0]\n\n    def t_shapes():\n        M = need(ns, \"ex6_models\")()\n        shapes = {k: tuple(m(Xte[:3]).shape) for k, m in M.items()}\n        ensure(shapes == {1: (3, 1), 2: (3, 1), 3: (3, 1), 4: (3, 8), 5: (3, 40, 8)}, f\"çıkış şekilleri {shapes}\")\n        cnt = {k: sum(p.numel() for p in m.parameters()) for k, m in M.items()}\n        ensure(cnt[1] == 41 and cnt[3] == 834 + 33 and cnt[4] == 952 + 32, \"PyTorch sayımı (iki bias)\")\n        T = need(ns, \"seq_targets\")(S[:2])\n        ensure(tuple(T.shape) == (2, 40, 8) and torch.equal(T[:, 39], S[:2, 40:, 0]) and torch.equal(T[:, 0, 0], S[:2, 1, 0]), \"dizi→dizi hedefleri\")\n    R_.test(\"modeller\", t_shapes)\n\n    def t_train():\n        t00 = time.perf_counter()\n        torch.manual_seed(0)\n        M = ns[\"ex6_models\"]()\n        tr = need(ns, \"train\")\n        res = {\"naif (1)\": F.mse_loss(Xte[:, -1], y1te).item(), \"naif (8)\": F.mse_loss(Xte[:, -1].expand(-1, 8), y8te).item()}\n        tr(M[1], Xtr, y1tr, 10, 5e-3)\n        tr(M[2], Xtr, y1tr, 10, 5e-3)\n        tr(M[3], Xtr, y1tr, 10, 1e-3)\n        lin8 = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(40, 8))\n        tr(lin8, Xtr, y8tr, 10, 5e-3)\n        tr(M[4], Xtr, y8tr, 10, 5e-3)\n        tr(M[5], Xtr, ns[\"seq_targets\"](S[:4000]), 10, 5e-3)\n        with torch.no_grad():\n            for k in (1, 2, 3):\n                res[k] = F.mse_loss(M[k](Xte), y1te).item()\n            res[\"it\"] = F.mse_loss(need(ns, \"iterative_forecast\")(M[3], Xte, 8), y8te).item()\n            res[\"lin8\"] = F.mse_loss(lin8(Xte), y8te).item()\n            res[4] = F.mse_loss(M[4](Xte), y8te).item()\n            res[5] = F.mse_loss(M[5](Xte)[:, -1], y8te).item()\n        rows = [[\"naif: son değer\", \"1\", f\"{res['naif (1)']:.4f}\"], [\"model 1: doğrusal (41 parametre)\", \"1\", f\"{res[1]:.4f}\"],\n                [\"model 2: tek rekürent nöron\", \"1\", f\"{res[2]:.4f}\"], [\"model 3: RNN 16-16-1\", \"1\", f\"{res[3]:.4f}\"],\n                [\"naif: son değeri 8 kez\", \"8\", f\"{res['naif (8)']:.4f}\"], [\"model 3 yinelemeli\", \"8\", f\"{res['it']:.4f}\"],\n                [\"doğrusal 40 → 8\", \"8\", f\"{res['lin8']:.4f}\"], [\"model 4: dizi→vektör\", \"8\", f\"{res[4]:.4f}\"], [\"model 5: dizi→dizi (son adım)\", \"8\", f\"{res[5]:.4f}\"]]\n        ensure(res[1] < res[\"naif (1)\"] / 3 and res[3] < res[\"naif (1)\"] / 3, \"doğrusal ve derin RNN naif tahmini açık farkla geçer\")\n        ensure(res[2] > res[1], \"tek nöronlu RNN doğrusal modeli geçemez\")\n        ensure(res[4] < res[\"lin8\"] and res[5] < res[\"lin8\"] and res[\"it\"] > res[5], \"8 adım: RNN'ler > doğrusal; yinelemeli < dizi→dizi\")\n        R_.table(f\"Ders alıştırması 6 (4000 seri, 10 epoch, {time.perf_counter() - t00:.0f} sn) — test MSE\", [\"model\", \"ufuk\", \"MSE\"], rows,\n                 note=\"Tek nöronun gizli durumu tek bir tanh'lı sayıdır: hem bellek hem çıkış aralığı (−1, 1) kısıtlı. Yinelemeli tahminde hatalar birikir; 8 değeri doğrudan tahmin etmek daha iyidir. Dizi→dizi her adımda kayıp aldığı için aynı epoch'ta en iyi sonuca yaklaşır (Géron: 0.008 → 0.006).\")\n    R_.test(\"eğitim ve karşılaştırma\", t_train)\n",
 "e": "import time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import need, ensure, fmt_time\n\n\ndef _series(n, steps, seed):\n    g = torch.Generator().manual_seed(seed)\n    f1, f2, o1, o2 = torch.rand(4, n, 1, generator=g)\n    t = torch.linspace(0, 1, steps)\n    s = 0.5 * torch.sin((t - o1) * (f1 * 10 + 10)) + 0.2 * torch.sin((t - o2) * (f2 * 20 + 20))\n    return (s + 0.1 * (torch.rand(n, steps, generator=g) - 0.5))[..., None]\n\n\ndef check(ns, R_):\n    def t_causal():\n        torch.manual_seed(0)\n        c = need(ns, \"CausalConv1d\")(1, 1, 3, 4)\n        x = torch.randn(1, 1, 30)\n        ensure(c(x).shape[-1] == 30, \"uzunluk korunur\")\n        x2 = x.clone()\n        x2[..., 20:] += 5\n        ensure(torch.allclose(c(x)[..., :20], c(x2)[..., :20]), \"gelecek geçmişi etkilemez\")\n        m = need(ns, \"WaveNet\")(16, (1, 2, 4, 8) * 2, 8)\n        ensure(tuple(m(torch.randn(2, 50, 1)).shape) == (2, 50, 8), \"çıkış (N, T, 8)\")\n        xi = torch.randn(1, 80, 1, requires_grad=True)\n        m(xi)[0, 60].sum().backward()\n        nz = torch.nonzero(xi.grad[0, :, 0])[:, 0]\n        ensure(int(nz.max()) == 60 and int(nz.min()) == 29, f\"t = 60 çıkışı tam [29, 60] girdilerine bağlı (RF 1 + 1 + 30 = 32), bulunan [{int(nz.min())}, {int(nz.max())}]\")\n    R_.test(\"nedensellik ve receptive field\", t_causal)\n\n    def t_train():\n        S = _series(5000, 48, 0)\n        Xtr, Xte, y8 = S[:4000, :40], S[4000:, :40], S[4000:, 40:, 0]\n        Ytr = torch.stack([S[:4000, k:k + 40, 0] for k in range(1, 9)], -1)\n        torch.manual_seed(0)\n        m = ns[\"WaveNet\"]()\n        opt = torch.optim.Adam(m.parameters(), 3e-3)\n        t0 = time.perf_counter()\n        for _ in range(10):\n            perm = torch.randperm(4000)\n            for i in range(0, 4000, 32):\n                idx = perm[i:i + 32]\n                loss = F.mse_loss(m(Xtr[idx]), Ytr[idx])\n                opt.zero_grad()\n                loss.backward()\n                opt.step()\n        with torch.no_grad():\n            mse = F.mse_loss(m(Xte)[:, -1], y8).item()\n        ensure(mse < 0.012, f\"WaveNet 8 adım MSE {mse:.4f}\")\n        rows = []\n        gru = torch.nn.GRU(1, 32, batch_first=True)\n        wn = ns[\"WaveNet\"](32, tuple(2 ** i for i in range(10)), 1)\n        for T in (256, 1024, 4096):\n            x = torch.randn(8, T, 1)\n            r = [T]\n            for mod in (gru, wn):\n                best = 1e9\n                for _ in range(2):\n                    t1 = time.perf_counter()\n                    out = mod(x)\n                    (out[0] if isinstance(out, tuple) else out).sum().backward()\n                    best = min(best, time.perf_counter() - t1)\n                r.append(fmt_time(best))\n            rows.append(r)\n        R_.note(f\"WaveNet dizi→dizi, 10 epoch ({time.perf_counter() - t0:.0f} sn): 8 adım MSE = {mse:.4f} (d aşamasındaki RNN'lerden iyi; RF = 32: 40'lık pencerenin çoğu).\")\n        R_.table(\"İleri + geri süresi, batch 8 (CPU): GRU-32 vs WaveNet-32 (1…512, RF 1024)\", [\"T\", \"GRU\", \"WaveNet\"], rows,\n                 note=\"GRU T adımı sırayla yürür (paralellik yok); nedensel konvolüsyonlar tüm zaman adımlarını aynı anda hesaplar. Bedeli: sabit, tasarımla belirlenen receptive field — ötesini hiç göremez.\")\n    R_.test(\"WaveNet eğitimi ve hız\", t_train)\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"seasonal_difference\")(np.arange(10.) ** 2, 3), (np.arange(3, 10.) ** 2 - np.arange(7.) ** 2), what=\"mevsimsel fark\")\n        close(need(ns, \"seasonal_naive\")(np.arange(10.), 3, 5), [7, 8, 9, 7, 8], what=\"son mevsimi tekrarla\")\n        y = np.add.outer(np.zeros(1), np.tile([1., 3, 2, 5, 4, 2, 1], 6))[0] + 0.5 * np.arange(42)\n        close(need(ns, \"sarima_lite\")(y, 7, 1, 7), np.tile([1., 3, 2, 5, 4, 2, 1], 1) + 0.5 * np.arange(42, 49), atol=1e-6, what=\"trend + mevsim tam tahmin edilir\")\n        rng = np.random.default_rng(0)\n        err = {\"naif\": [], \"mevsimsel naif\": [], \"SARIMA-benzeri\": []}\n        for _ in range(60):\n            n, h = 140, 14\n            t = np.arange(n + h)\n            pattern = rng.normal(0, 3, 7)\n            e = np.zeros(n + h)\n            for i in range(1, n + h):\n                e[i] = 0.7 * e[i - 1] + rng.normal(0, 0.5)\n            y = 10 + 0.3 * t + pattern[t % 7] + e\n            tr, te = y[:n], y[n:]\n            err[\"naif\"].append(np.abs(np.full(h, tr[-1]) - te).mean())\n            err[\"mevsimsel naif\"].append(np.abs(ns[\"seasonal_naive\"](tr, 7, h) - te).mean())\n            err[\"SARIMA-benzeri\"].append(np.abs(ns[\"sarima_lite\"](tr, 7, 2, h) - te).mean())\n        m = {k: float(np.mean(v)) for k, v in err.items()}\n        ensure(m[\"SARIMA-benzeri\"] < m[\"mevsimsel naif\"] < m[\"naif\"], f\"MAE sırası {m}\")\n        R_.table(\"Trend + haftalık mevsim + AR(1) gürültü, 60 seri, 14 gün ileri\", [\"yöntem\", \"MAE\"], [[k, f\"{v:.2f}\"] for k, v in m.items()],\n                 note=\"Mevsimsel naif (Géron'un yolcu verisindeki 'geçen haftanın aynı günü') naif tahmini rahat geçer ama trendi kaçırır (her hafta 7 × 0.3 geride). (1 − B)(1 − B⁷) farkı hem trendi hem mevsimi siler; kalan AR yapı birkaç gün ileri bilgi verir. Tam SARIMA bu iskeletin üstüne MA ve mevsimsel AR/MA terimleri ekler.\")\n    R_.test(\"mevsimsel tahmin\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"ensemble_stats\", \"wss_gap\", \"time_mean\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        R, T = 4000, 60\n        t = np.arange(T)\n        w = 0.3\n        procs = {\n            \"rastgele fazlı kosinüs\": np.cos(w * t + rng.uniform(0, 2 * np.pi, (R, 1))),\n            \"rastgele genlikli kosinüs\": rng.normal(size=(R, 1)) * np.cos(w * t),\n            \"rastgele yürüyüş\": np.cumsum(rng.normal(size=(R, T)), 1),\n            \"rastgele sabit X_t = A\": rng.normal(size=(R, 1)) * np.ones(T),\n        }\n        e = rng.normal(size=(R, T + 200))\n        a = np.zeros_like(e)\n        for i in range(1, T + 200):\n            a[:, i] = 0.8 * a[:, i - 1] + e[:, i]\n        procs[\"AR(1), φ = 0.8\"] = a[:, 200:]\n        m, K = need(ns, \"ensemble_stats\")(procs[\"rastgele genlikli kosinüs\"])\n        close(K[5, 9], np.mean(procs[\"rastgele genlikli kosinüs\"][:, 5] * procs[\"rastgele genlikli kosinüs\"][:, 9]) - m[5] * m[9], rtol=1e-9, what=\"K tanımı\")\n        rows, gaps = [], {}\n        for name, X in procs.items():\n            g = need(ns, \"wss_gap\")(X)\n            tm = need(ns, \"time_mean\")(X)\n            gaps[name] = g\n            rows.append([name, f\"{g[0]:.3f}\", f\"{g[1]:.3f}\", \"evet\" if max(g) < 0.15 else \"hayır\", f\"{np.std(tm):.3f}\"])\n        ok = {k: max(v) < 0.15 for k, v in gaps.items()}\n        ensure(ok == {\"rastgele fazlı kosinüs\": True, \"rastgele genlikli kosinüs\": False, \"rastgele yürüyüş\": False, \"rastgele sabit X_t = A\": True, \"AR(1), φ = 0.8\": True}, f\"WSS sınıflandırması {ok}\")\n        R_.table(f\"{R} gerçekleşme × {T} adım\", [\"süreç\", \"ortalama sapması\", \"kovaryans sapması\", \"WSS?\", \"zaman ortalamalarının std'si\"], rows,\n                 note=\"Rastgele genlikli kosinüsün ortalaması sabit (0) ama K(t₁, t₂) = cos ωt₁ cos ωt₂ fark yerine mutlak zamana bağlı → WSS değil. Rastgele yürüyüşün varyansı t ile büyür. 'X_t = A' WSS'dir ama ergodik değildir: tek bir gerçekleşmenin zaman ortalaması A'dır, topluluk ortalaması 0 — tek seriden istatistik öğrenen her model (ARMA, RNN) örtük olarak ergodikliğe güvenir.\")\n    R_.test(\"durağanlık ve ergodiklik\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        n = 12000\n        phi = np.where(np.arange(n)[:, None] < 6000, [0.6, 0.3], [-0.5, 0.3])\n        y = np.zeros(n)\n        e = rng.normal(size=n)\n        for i in range(2, n):\n            y[i] = phi[i, 0] * y[i - 1] + phi[i, 1] * y[i - 2] + e[i]\n        pred, W = need(ns, \"nlms\")(y, 2, 0.05)\n        close(W[4000:6000].mean(0), [0.6, 0.3], atol=0.1, what=\"ilk rejimde ağırlıklar (ortalama)\")\n        close(W[-2000:].mean(0), [-0.5, 0.3], atol=0.1, what=\"ikinci rejime uyum (ortalama)\")\n        X = np.stack([y[1:5999], y[:5998]], 1)\n        fixed = np.linalg.lstsq(X, y[2:6000], rcond=None)[0]\n        seg = slice(7000, n)\n        idx = np.arange(n)[seg]\n        fx = fixed[0] * y[idx - 1] + fixed[1] * y[idx - 2]\n        oracle = phi[idx, 0] * y[idx - 1] + phi[idx, 1] * y[idx - 2]\n        m = {\"sabit AR(2) (ilk yarıya uydurulmuş)\": np.mean((y[seg] - fx) ** 2), \"NLMS μ = 0.05\": np.mean((y[seg] - pred[seg]) ** 2), \"gerçek katsayılar (alt sınır)\": np.mean((y[seg] - oracle) ** 2)}\n        ensure(m[\"NLMS μ = 0.05\"] < 0.8 * m[\"sabit AR(2) (ilk yarıya uydurulmuş)\"], \"NLMS rejim değişimini izler\")\n        rows = [[k, f\"{v:.3f}\"] for k, v in m.items()]\n        for mu in (0.005, 0.2, 0.8):\n            p2, _ = ns[\"nlms\"](y, 2, mu)\n            rows.append([f\"NLMS μ = {mu}\", f\"{np.mean((y[seg] - p2[seg]) ** 2):.3f}\"])\n        R_.table(\"AR(2) katsayıları t = 6000'de değişiyor; 7000 sonrası tek adım MSE\", [\"öngörücü\", \"MSE\"], rows,\n                 note=\"Sabit parametreli model (ARMA, eğitilmiş RNN) durağanlık varsayar; rejim değişince yanlış katsayılarla tahmin etmeye devam eder. LMS ailesi her adımda gradyan adımı atarak izler. μ küçükse yavaş uyum, büyükse gürültülü ağırlıklar (gecikme–gürültü ödünleşimi); NLMS adımı girdi gücüne normalize ederek kararlılığı μ ∈ (0, 2) aralığında garanti eder.\")\n    R_.test(\"uyarlamalı filtre\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_11_timeseries as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Rastgele süreç** indekslenmiş rastgele değişkenler topluluğudur; bir ölçümü (gerçekleşmesi) bir **zaman serisidir** — rastgele değişkenle sayı arasındaki ilişki gibi. Süreci tam tanımlamak için tüm zamanların ortak dağılımı gerekir. **Katı durağanlık:** ortak dağılım zamanda kaydırmayla değişmez. **Geniş anlamda durağanlık (WSS):** ortalama sabit, otokovaryans yalnız zaman farkına bağlı ($K(t_1,t_2)=K(t_1-t_2)$), ikinci moment sonlu. ARMA, RNN ve CNN'lerin parametreleri sabittir — durağanlık varsayımı bu yüzden önemlidir; **uyarlamalı filtreler** parametrelerini güncelleyerek durağan olmayan sinyalleri izler.

**ARMA (Wold, 1930'lar):** $\hat y_t=\sum_{i=1}^p\alpha_iy_{t-i}+\sum_{i=1}^q\theta_i\epsilon_{t-i}$, $\epsilon_t=y_t-\hat y_t$ — geçmiş değerlerin ağırlıklı toplamı (otoregresif) + geçmiş tahmin hatalarının ağırlıklı toplamı (hareketli ortalama). Seri durağan değilse **fark alma**: bir adımlık fark türevin yaklaşığıdır ve doğrusal trendi sabite çevirir; d tur fark d. dereceden polinom trendleri siler (d: integrasyon derecesi). **ARIMA** (Box & Jenkins 1970) d kez farklayıp ARMA uygular, tahmine çıkarılanları geri ekler. **SARIMA** aynı yaklaşımı $t-s, t-2s, …$ gecikmelerine (P, D, Q, s) de uygular: toplam yedi hiperparametre.

**RNN ile tahmin** (Géron, alıştırma 6): veriler `[batch, zaman, boyut]` şeklinde. Önce temel çizgiler: **naif tahmin** (son değer; şaşırtıcı derecede zor geçilir) ve doğrusal model. Tek rekürent nöronun gizli durumu tek bir sayıdır — doğrusal modeli geçemez. Çok adımlı tahmin üç yolla: (1) tek adımlık modeli kendi tahminiyle besleyerek yinelemek (hatalar birikir), (2) son adımda 8 değeri birden üreten dizi→vektör, (3) her adımda sonraki 8 değeri tahmin eden **dizi→dizi** (her adımda kayıp → daha kararlı, daha hızlı eğitim; model nedensel). Uzun diziler için 1B CNN'ler ve **WaveNet**: genişletmesi her katmanda ikiye katlanan nedensel konvolüsyonlar.

## 1 · Elle

AR(2): 1.2 + 0.3 = 1.5. ARMA(1,1): 1.0 + 0.2 = 1.2. ARIMA: fark [2, 2, 2, 2], tahmin edilen fark 2 → 11 + 2 = 13.

Yule–Walker: $\rho_k=\phi_1\rho_{k-1}+\phi_2\rho_{k-2}$ ($\rho_0=1$, $\rho_{-k}=\rho_k$). $k=1$: $\rho_1=\phi_1+\phi_2\rho_1\Rightarrow\rho_1=\phi_1/(1-\phi_2)=0.6/0.7=0.857$. $k=2$: $\rho_2=0.6\cdot0.857+0.3=0.814$. Tersine, örnek otokorelasyonlardan $\phi$ çözülür: Yule–Walker tahmincisi (c aşaması).

AR(1) varyansı: $\gamma_0=\phi^2\gamma_0+\sigma^2\Rightarrow\gamma_0=\sigma^2/(1-\phi^2)=1/0.36=2.78$; $|\phi|\to1$'de patlar (birim kök: rastgele yürüyüş, durağan değil).

Parametreler (Keras SimpleRNN: $W_x$, $W_h$, tek $b$): $1\cdot16+256+16=288$; $16\cdot16+256+16=528$; $16+1+1=18$ → **834**; yoğun 8: $16\cdot8+8=136$ → **952**. PyTorch `nn.RNN` iki bias tutar ($b_{ih}$, $b_{hh}$): her katmana +h. Model 1: 41, model 2: 3 (Keras).

WaveNet: $1+2\cdot(2^{10}-1)\cdot(k-1)=2047$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**ARMA / ARIMA tahminleri**

- `ar2`: AR(2): $\hat y_t=0.6y_{t-1}+0.3y_{t-2}$, $y_{t-1}=2$, $y_{t-2}=1$
- `arma`: ARMA(1,1): $\alpha=0.5$, $\theta=0.4$, $y_{t-1}=2$, $\epsilon_{t-1}=0.5$
- `arima`: [3, 5, 7, 9, 11] serisinde d = 1 ARIMA, farklanmış seri sabit tahmin ediliyor: sonraki değer

**Yule–Walker, AR(2) $\phi=(0.6, 0.3)$ ve AR(1)**

- `rho1`: $\rho_1$
- `rho2`: $\rho_2$
- `var1`: AR(1) $\phi=0.8$, $\sigma^2=1$: durağan varyans

**Ders alıştırması 6 (Keras sayımı: SimpleRNN tek bias) ve WaveNet**

- `m3`: Model 3: RNN 16 → RNN 16 → RNN 1, tek değişkenli girdi
- `m4`: Model 4: RNN 16 → RNN 16 → yoğun 8
- `rf`: WaveNet: k = 2, genişletmeler 1, 2, …, 512 iki kez yığılmış: receptive field

In [ ]:
cevaplar = dict(
    ar2=None,
    arma=None,
    arima=None,
    rho1=None,
    rho2=None,
    var1=None,
    m3=None,
    m4=None,
    rf=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"ar2":[1.5,1e-9],"arma":[1.2,1e-9],"arima":[13,0],"rho1":[0.8571428571428571,0.000001],"rho2":[0.8142857142857143,0.000001],"var1":[2.7777777777777777,0.000001],"m3":[834,0],"m4":[952,0],"rf":[2047,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `difference(y, d=1)` — d tur fark.
- `undifference(y, dz, d)` — `y` gözlenmiş seri, `dz` d kez farklanmış serinin gelecek tahminleri → `y`'nin gelecek tahminleri (özyineleme: bir alt mertebeye in, sonra kümülatif topla).
- `arma_predict(y, alpha, theta)` — her t = 0..n için bir adımlık tahmin (n + 1 eleman, sonuncusu geleceğe); başlangıçtan önceki y ve ε sıfır.
- `ar_acf(phi, nlags)` — AR(p)'nin kuramsal otokorelasyonları $\rho_0..\rho_{nlags}$: önce $\rho_1..\rho_p$ için p × p Yule–Walker sistemini (Gauss eleme) çöz, sonra özyinele.
- `rnn_params(sizes, n_in=1, double_bias=False)` — yığılmış basit RNN katmanlarının parametre sayısı (Keras: tek bias; PyTorch: iki).
- `receptive_field(k, dilations)`.

In [ ]:
%%writefile stage_b.py
def difference(y, d=1):
    ...


def undifference(y, dz, d):
    ...


def arma_predict(y, alpha, theta):
    ...


def ar_acf(phi, nlags):
    ...


def rnn_params(sizes, n_in=1, double_bias=False):
    ...


def receptive_field(k, dilations):
    ...


print(difference([1, 4, 9, 16, 25, 36], 2))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy.

- `acf(y, maxlag)` — örnek otokorelasyon (ortalama çıkarılmış, $\hat\gamma_k=\frac1n\sum(y_t-\bar y)(y_{t+k}-\bar y)$, $\hat\gamma_k/\hat\gamma_0$); döngüsüz.
- `lag_matrix(y, p)` → `(X, t)`: satır t için `[y_{t-1}, …, y_{t-p}]`, hedef `y_t`; döngüsüz.
- `fit_ar_ls(y, p)` (en küçük kareler) ve `fit_ar_yw(y, p)` (Toeplitz $R\phi=r$); döngüsüz.
- `pacf(y, maxlag)` — gecikme k'daki kısmi otokorelasyon = AR(k) Yule–Walker uyumunun son katsayısı (tek döngü).
- `forecast_ar(y, phi, h)` — yinelemeli h adım (tek döngü).

In [ ]:
%%writefile stage_c.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def acf(y, maxlag):
    ...


def lag_matrix(y, p):
    ...


def fit_ar_ls(y, p):
    ...


def fit_ar_yw(y, p):
    ...


def pacf(y, maxlag):
    ...


def forecast_ar(y, phi, h):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch, ders alıştırması 6. `generate_time_series(n, steps, seed)` verilir (iki rastgele sinüs + gürültü; Géron). 5000 seri: 4000 eğitim, 1000 test; ilk 40 değer girdi.

- `ex6_models()` → `{1: doğrusal (40 → 1), 2: RNN(1), 3: RNN16 → RNN16 → RNN1, 4: RNN16 → RNN16 → yoğun 8 (son adım), 5: RNN16 → RNN16 → her adımda yoğun 8}`. Çıkışlar: modeller 1–3 `(N, 1)`, 4 `(N, 8)`, 5 `(N, 40, 8)`. RNN çıkışı son katmanın son adımı (model 2, 3).
- `seq_targets(S, n_steps=40, ahead=8)` — model 5 hedefi `(N, 40, 8)`: t adımında `S[t+1 … t+8]`.
- `train(m, X, Y, epochs=10, lr=1e-3)` — Adam, MSE, batch 32.
- `iterative_forecast(m, X, h, window=40)` — tek adımlık modeli kendi tahminiyle besle.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn as nn
import torch.nn.functional as F


def generate_time_series(n, steps, seed):
    g = torch.Generator().manual_seed(seed)
    f1, f2, o1, o2 = torch.rand(4, n, 1, generator=g)
    t = torch.linspace(0, 1, steps)
    s = 0.5 * torch.sin((t - o1) * (f1 * 10 + 10)) + 0.2 * torch.sin((t - o2) * (f2 * 20 + 20))
    s = s + 0.1 * (torch.rand(n, steps, generator=g) - 0.5)
    return s[..., None]


def ex6_models():
    ...


def seq_targets(S, n_steps=40, ahead=8):
    ...


def train(m, X, Y, epochs=10, lr=1e-3):
    ...


def iterative_forecast(m, X, h, window=40):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `CausalConv1d(cin, cout, k, d=1)` — solda $(k-1)d$ dolgu.
- `WaveNet(c=16, dilations=(1,2,4,8)*2, out=8)` — giriş nedensel konv. (k = 2), her blokta kapılı aktivasyon $\tanh(W_f*x)\odot\sigma(W_g*x)$, 1×1 artık ve atlama; atlamaların toplamı → ReLU → 1×1 → ReLU → 1×1. Girdi `(N, T, 1)`, çıkış `(N, T, out)`.
- Kontroller: nedensellik (Jacobian), receptive field (gradyan desteği), dizi→dizi eğitimi ve GRU ile uzun dizi süre karşılaştırması.

In [ ]:
%%writefile stage_e.py
import time
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalConv1d(nn.Conv1d):
    def __init__(self, cin, cout, k, d=1):
        ...

    def forward(self, x):
        ...


class WaveNet(nn.Module):
    def __init__(self, c=16, dilations=(1, 2, 4, 8) * 2, out=8):
        super().__init__()
        ...

    def forward(self, x):
        ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Mevsimsellik: mevsimsel naif tahmin ve SARIMA-benzeri model

NumPy.

- `seasonal_difference(y, s)` — $y_t-y_{t-s}$.
- `seasonal_naive(y, s, h)` — $\hat y_{T+k}=y_{T+k-s\lceil k/s\rceil}$ (son mevsimi tekrarla).
- `sarima_lite(y, s, p, h)` — $z=(1-B)(1-B^s)y$, z'ye AR(p) (en küçük kareler) uydur, h adım yinele, sonra $y_t=z_t+y_{t-1}+y_{t-s}-y_{t-s-1}$ ile geri kur.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def seasonal_difference(y, s):
    ...


def seasonal_naive(y, s, h):
    ...


def sarima_lite(y, s, p, h):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Durağanlık ve ergodiklik: topluluk ortalaması vs zaman ortalaması

NumPy, döngüsüz. `X (gerçekleşme, T)`: aynı sürecin birçok örneği.

- `ensemble_stats(X)` → `(m (T,), K (T, T))`: topluluk ortalaması ve otokovaryansı.
- `wss_gap(X)` → `(ortalama sapması, kovaryans sapması)`: $\max_t|m_t-\bar m|/\sqrt{\bar K_0}$ ve $\max|K(t_1,t_2)-\overline K(t_1-t_2)|/\bar K_0$ — köşegen ortalamaları `bincount` ile.
- `time_mean(X)` — her gerçekleşmenin zaman ortalaması `(gerçekleşme,)`.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def ensemble_stats(X):
    ...


def wss_gap(X):
    ...


def time_mean(X):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Durağan olmayan seriler için uyarlamalı filtre (NLMS)

NumPy (zaman döngüsü serbest).

- `nlms(y, p, mu=0.1, eps=1e-6)` — normalize LMS tek adım öngörücü: $\hat y_t=w^\top u_t$, $u_t=[y_{t-1},…,y_{t-p}]$, $w\leftarrow w+\mu\,e_tu_t/(\epsilon+\|u_t\|^2)$. `(tahminler (n,), ağırlık geçmişi (n, p))`; ilk p tahmin 0.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def nlms(y, p, mu=0.1, eps=1e-6):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: van den Oord ve ark. (2016): WaveNet

[van den Oord ve ark. — WaveNet: A Generative Model for Raw Audio (2016)](https://arxiv.org/abs/1609.03499)

e aşamasındaki WaveNet'i otoregresif üretici modele çevir: μ-law kodlaması (256 seviye, Denk. 1) ve softmax çıktı (Bölüm 2.2), kapılı aktivasyon, artık + atlama bağlantıları; küçük bir ses veri setinde (ör. tek konuşmacının birkaç dakikası ya da sentetik nota dizileri) eğit, örnekleme hızını ölç ve önbellekli hızlı üretimi (Paine ve ark. 2016'daki "fast WaveNet" kuyrukları) uygulayıp yavaş üretimle karşılaştır.


**Adımlar**

1. μ-law: $f(x)=\mathrm{sign}(x)\ln(1+\mu|x|)/\ln(1+\mu)$.
2. Hızlı üretim: her katman için son d adımı tutan dairesel tampon.

**Kabul ölçütleri**

- Üretilen ses/dizi eğitim verisinin istatistiklerini taşır (spektrum veya nota dağılımı).
- Önbellekli üretim yavaşıyla bire bir aynı örnekleri verir ve katman sayısıyla doğrusal hızlanır.

## Makale ödevi: Oreshkin ve ark. (2020): N-BEATS

[Oreshkin, Carpov, Chapados, Bengio — N-BEATS: Neural basis expansion analysis for interpretable time series forecasting (ICLR 2020)](https://arxiv.org/abs/1905.10437)

Geri/ileri artık (backcast/forecast) yığınlı N-BEATS'in hem genel hem yorumlanabilir (trend: polinom tabanı, mevsimsellik: Fourier tabanı) sürümünü uygula; x1'deki mevsimsel sentetik serilerde ve ders alıştırması 6 verisinde SARIMA-benzeri modelle, d aşamasındaki dizi→dizi RNN'le ve e'deki WaveNet'le sMAPE / MASE üzerinden karşılaştır.


**Adımlar**

1. MASE paydası: mevsimsel naif tahminin eğitim içi ortalama mutlak hatası.

**Kabul ölçütleri**

- Yorumlanabilir sürümün trend ve mevsim yığınları ayrık bileşenleri ayırır (çizimle).
- N-BEATS mevsimsel veride naif ve SARIMA-benzeri modeli geçer.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 15](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Box, Jenkins, Reinsel, Ljung — Time Series Analysis: Forecasting and Control](https://www.wiley.com/en-us/Time+Series+Analysis%3A+Forecasting+and+Control%2C+5th+Edition-p-9781118675021) — ©
- [Hyndman & Athanasopoulos — Forecasting: Principles and Practice (3. baskı)](https://otexts.com/fpp3/) — CC BY-NC-ND
- [van den Oord ve ark. — WaveNet: A Generative Model for Raw Audio (2016)](https://arxiv.org/abs/1609.03499) — arXiv
- [Oreshkin ve ark. — N-BEATS (ICLR 2020)](https://arxiv.org/abs/1905.10437) — arXiv
- [Haykin — Adaptive Filter Theory (LMS / NLMS)](https://www.pearson.com/en-us/subject-catalog/p/adaptive-filter-theory/P200000003275) — ©
- [d2l.ai — 9.1 Working with Sequences (otoregresif modeller)](https://d2l.ai/chapter_recurrent-neural-networks/sequence.html) — CC-BY-SA 4.0